# Batch Normalization (Ioffe & Szegedy, 2015)
arXiv:1502.03167 — generated by ArXivist

Reproduces the MNIST experiment from Section 4.1.

In [ ]:
!pip install torch torchvision -q

In [ ]:
import torch, torch.nn as nn
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Batch Normalizing Transform (Algorithm 1)

$$\mu_\mathcal{B} = \frac{1}{m}\sum_{i=1}^m x_i \qquad \sigma^2_\mathcal{B} = \frac{1}{m}\sum_{i=1}^m (x_i - \mu_\mathcal{B})^2$$

$$\hat{x}_i = \frac{x_i - \mu_\mathcal{B}}{\sqrt{\sigma^2_\mathcal{B} + \epsilon}} \qquad y_i = \gamma \hat{x}_i + \beta$$

In [ ]:
class BatchNorm1dManual(nn.Module):
    def __init__(self, num_features, epsilon=1e-5, momentum=0.9):
        super().__init__()
        self.epsilon = epsilon
        self.momentum = momentum
        self.gamma = nn.Parameter(torch.ones(num_features))
        self.beta = nn.Parameter(torch.zeros(num_features))
        self.register_buffer('running_mean', torch.zeros(num_features))
        self.register_buffer('running_var', torch.ones(num_features))

    def forward(self, x):
        if self.training:
            mu = x.mean(0)
            var = x.var(0, unbiased=False)
            self.running_mean = self.momentum * self.running_mean + (1 - self.momentum) * mu
            self.running_var = self.momentum * self.running_var + (1 - self.momentum) * var
        else:
            mu, var = self.running_mean, self.running_var
        x_hat = (x - mu) / torch.sqrt(var + self.epsilon)
        return self.gamma * x_hat + self.beta

bn = BatchNorm1dManual(4)
x = torch.randn(8, 4)
y = bn(x)
print('Input shape:', x.shape, '| Output shape:', y.shape)
print('Output mean (should be ~0):', y.mean(0).detach().round(decimals=3))
print('Output std (should be ~1):', y.std(0).detach().round(decimals=3))

## BN-MLP: 3-layer sigmoid network (Section 4.1)

Architecture: Input(784) → [Linear → BN → Sigmoid] × 3 → Linear(10)

In [ ]:
class BatchNormMLP(nn.Module):
    def __init__(self, input_dim=784, hidden=100, num_classes=10):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(input_dim if i==0 else hidden, hidden, bias=False),
                          BatchNorm1dManual(hidden), nn.Sigmoid())
            for i in range(3)
        ])
        self.out = nn.Linear(hidden, num_classes)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return self.out(x)

model = BatchNormMLP().to(device)
print('Params:', sum(p.numel() for p in model.parameters()))
x_test = torch.randn(60, 784).to(device)
print('Forward pass output shape:', model(x_test).shape)

## Mini training loop on synthetic data (no download needed)

In [ ]:
import torch.optim as optim
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

for step in range(20):
    x = torch.randn(60, 784).to(device)
    y = torch.randint(0, 10, (60,)).to(device)
    optimizer.zero_grad()
    loss = criterion(model(x), y)
    loss.backward()
    optimizer.step()
    if step % 5 == 0:
        print(f'step {step}: loss={loss.item():.4f}')
print('Mini loop complete — setup verified.')

## Full MNIST training (uncomment to run)

In [ ]:
# from torchvision import datasets, transforms
# from torch.utils.data import DataLoader
# transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])
# train_data = datasets.MNIST('./data', train=True, download=True, transform=transform)
# test_data = datasets.MNIST('./data', train=False, download=True, transform=transform)
# train_loader = DataLoader(train_data, batch_size=60, shuffle=True)
# test_loader = DataLoader(test_data, batch_size=60)
# step = 0
# for epoch in range(100):
#     for x, y in train_loader:
#         x, y = x.to(device), y.to(device)
#         optimizer.zero_grad()
#         loss = criterion(model(x), y)
#         loss.backward()
#         optimizer.step()
#         step += 1
#         if step >= 50000: break
#     if step >= 50000: break
# model.eval()
# correct = sum((model(x.to(device)).argmax(1) == y.to(device)).sum().item() for x, y in test_loader)
# print(f'Test accuracy: {correct/10000:.4f}')

## Paper's reported results

| Metric | Value |
|---|---|
| BN-x5 speedup vs Inception | 14x fewer steps |
| BN-Inception ensemble top-5 error | 4.9% |

After running full MNIST training, bring your test accuracy back to trigger Stage 6.